# 2. Transcription and speaker separation

**Speaker separation** comes from the dataset's split channels: each file holds one speaker. No diarization is needed.
**Role labels** come from channel metadata, with an agent-phrase cross-check.
**Turn boundaries** are predicted from ASR word timestamps, not taken from the dataset.
Reference transcripts are kept apart and used only for evaluation.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, Markdown, display

from calltrace.config import PROJECT_ROOT, load_config
from calltrace.data import AppTekDataset

pd.set_option("display.max_colwidth", 160, "display.max_rows", 80)
config = load_config()
dataset = AppTekDataset(config)
results = config.path("results")
evaluation_dir = results / "evaluation"


## Choosing the ASR model
On a pilot call that is not part of the sample. The published benchmark (P1) reports Parakeet v3 at 9.2% and Whisper Large v3 at 15.0% WER with this segmentation.

In [2]:
pilot = pd.read_json(evaluation_dir / "asr_pilot_comparison.jsonl", lines=True)
pilot["wer_pct"] = (100 * pilot.wer).round(2)
pilot[["model", "wer_pct", "reference_words", "substitutions", "deletions", "insertions", "load_s", "inference_s", "audio_s_two_channels"]]

,model,wer_pct,reference_words,substitutions,deletions,insertions,load_s,inference_s,audio_s_two_channels
0,"nemo-parakeet-tdt-0.6b-v3 (int8, CPU)",6.02,1180,34,20,17,4.0,61.2,1208.8
1,"faster-whisper large-v3 (cpu, int8)",7.71,1180,33,42,16,789.6,477.1,1208.8
2,"nemo-parakeet-tdt-0.6b-v3 (full precision, CPU)",4.83,1180,27,9,21,7.1,78.4,1208.8


One call is too small to rank models on its own. It supports the published result and shows the local scoring works.

**This pilot also hid a problem.** The first full run used int8 weights: the pilot looked fine, yet one sampled call reached 39% WER.
The next cell shows how the cause was found. Full-precision weights are now the configured default.

In [3]:
vad = pd.read_csv(evaluation_dir / "asr_deletion_diagnosis.csv", dtype={"call_id": str, "channel": str})
print("Hypothesis 1: VAD or low level. WER % with int8 weights:")
display(vad.pivot_table(index=["call_id", "channel"], columns="variant", values="wer_pct"))
whisper = pd.read_csv(evaluation_dir / "asr_hard_channels_whisper.csv", dtype={"call_id": str, "channel": str})
precision = pd.read_csv(evaluation_dir / "asr_parakeet_precision.csv", dtype={"call_id": str, "channel": str})
compare = vad[vad.variant == "default_vad"][["call_id", "channel", "wer_pct"]].rename(columns={"wer_pct": "parakeet_int8_wer_pct"})
print("Hypotheses 2 and 3: the voices, or the int8 weights:")
compare.merge(whisper[["call_id", "channel", "whisper_wer_pct"]], how="inner").merge(
    precision[["call_id", "channel", "parakeet_full_precision_wer_pct"]], how="inner")

Hypothesis 1: VAD or low level. WER % with int8 weights:


variant          default_vad  default_vad_normalised  sensitive_vad  \
call_id channel                                                       
001     2               2.74                    2.88           2.59   
003     1              45.78                   47.53          46.08   
        2              28.48                   33.41          39.01   
008     2              36.18                   35.14          40.57   
009     2              43.36                   45.61          37.34   
010     2              52.06                   54.30          54.01   
012     1               3.87                    3.87           4.17   

variant          sensitive_vad_normalised  
call_id channel                            
001     2                            3.03  
003     1                           47.53  
        2                           39.24  
008     2                           33.59  
009     2                           39.10  
010     2                           54.74  
012     1                            4.17

Hypotheses 2 and 3: the voices, or the int8 weights:


,call_id,channel,parakeet_int8_wer_pct,whisper_wer_pct,parakeet_full_precision_wer_pct
0,003,1,45.78,6.83,5.38
1,010,2,52.06,36.59,35.65
2,009,2,43.36,17.54,18.30


## Run transcription
This loads cached results. On a clean checkout it would download the audio and run ASR for each channel.

In [4]:
from calltrace.pipeline import CallPipeline

records = dataset.load_manifest()["selected"]
pipeline = CallPipeline(config, results, use_llm=False)
run_summary = pipeline.run(records)
run_summary

[001] already complete, skipped
[002] already complete, skipped
[003] already complete, skipped
[004] already complete, skipped
[005] already complete, skipped
[006] already complete, skipped
[007] already complete, skipped
[008] already complete, skipped
[009] already complete, skipped
[010] already complete, skipped
[011] already complete, skipped
[012] already complete, skipped
[013] already complete, skipped
[014] already complete, skipped


{'processed': [],
 'failed': [],
 'resumed': ['001',
  '002',
  '003',
  '004',
  '005',
  '006',
  '007',
  '008',
  '009',
  '010',
  '011',
  '012',
  '013',
  '014']}

## A segmented transcript
Each turn keeps a stable ID, timestamps, speaker, role, raw ASR, and cleaned text.

In [5]:
from calltrace.export import load_processed_calls

processed = load_processed_calls(results / "calls")
example = processed[0][0]
print(example.speaker_boundaries_source)
pd.DataFrame([t.model_dump() for t in example.turns])[["turn_id", "start", "end", "speaker", "role", "text_raw", "is_backchannel"]].head(14)

Speaker identity: dataset split channels. Turn boundaries: predicted from ASR word timestamps.


,turn_id,start,end,speaker,role,text_raw,is_backchannel
0,001-T0001,13.346,17.966,Speaker 1,Agent,"Hi, thanks for calling A and Z. You are speaking with Tom. How can I help you today?",False
1,001-T0002,18.354,37.314,Speaker 2,Customer,"Hey Tom, uh my name's Laura. I've just been on the banking app and I've made a mess of things to be honest. I went onto the app, I've opened it up like norm...",False
2,001-T0003,37.314,50.590,Speaker 2,Customer,So I've sent all this money and I'm really worried because obviously I didn't want to go to the wrong place and yeah. I don't know what to do because it's q...,False
3,001-T0004,51.474,70.894,Speaker 1,Agent,"Okay, um I can understand how frustrating that's been. We've all been there. Just another manic Monday, as they say. Uh just so I can bring up your details,...",False
4,001-T0005,67.682,74.462,Speaker 2,Customer,"Yeah, it's that's okay. It's uh E R T.",False
5,001-T0006,74.194,79.294,Speaker 1,Agent,"Okay, E R T. Perfect. And just a year of birth, thanks.",False
6,001-T0007,80.082,83.182,Speaker 2,Customer,Uh it's nineteen seventy-seven.,False
7,001-T0008,83.858,92.718,Speaker 1,Agent,Perfect 1977. Great. All right. So I have got your details up here. Could you please tell me which transaction this was?,False
8,001-T0009,93.122,103.918,Speaker 2,Customer,"So I just did the transaction before. So it was just my last one and it was meant to go to my rent, but I've sent it to a friend.",False
9,001-T0010,103.874,107.214,Speaker 1,Agent,Oh God. And so have you already tried to contact the friend?,False


## Role assignment and turn statistics

In [6]:
rows = []
for transcript, _ in processed:
    agent = next(s for s in transcript.speakers if s.role == "Agent")
    rows.append({"call_id": transcript.call_id, "agent_channel": agent.channel, "role_method": agent.role_method,
                 "agent_phrase_check": agent.role_text_check, "turns": len(transcript.turns),
                 "backchannels": sum(t.is_backchannel for t in transcript.turns),
                 "low_confidence_turns": sum(t.low_confidence for t in transcript.turns)})
pd.DataFrame(rows)

,call_id,agent_channel,role_method,agent_phrase_check,turns,backchannels,low_confidence_turns
0,001,1,dataset metadata for this channel file,agrees (agent cues: 4 vs 1),59,5,1
1,002,1,dataset metadata for this channel file,agrees (agent cues: 3 vs 0),41,0,0
2,003,1,dataset metadata for this channel file,agrees (agent cues: 5 vs 0),39,7,0
3,004,1,dataset metadata for this channel file,agrees (agent cues: 2 vs 0),101,22,1
4,005,1,dataset metadata for this channel file,agrees (agent cues: 1 vs 0),38,4,0
5,006,2,dataset metadata for this channel file,agrees (agent cues: 1 vs 0),51,3,1
6,007,1,dataset metadata for this channel file,agrees (agent cues: 2 vs 0),92,27,0
7,008,1,dataset metadata for this channel file,agrees (agent cues: 2 vs 1),68,5,2
8,009,1,dataset metadata for this channel file,agrees (agent cues: 2 vs 0),51,4,1
9,010,1,dataset metadata for this channel file,agrees (agent cues: 2 vs 0),125,28,6


## Unclear audio
- Unreadable or empty audio raises an error, which is recorded per call in `failures.jsonl`; the run continues.
- Parakeet log-probabilities flag turns as `low_confidence`. This is an uncalibrated heuristic for review, not a probability of correctness.
- A silent channel or a channel with no recognised speech produces a quality note.

In [7]:
failure_file = results / "failures.jsonl"
print("Failures logged:", failure_file.read_text(encoding="utf-8") if failure_file.exists() else "none")
pd.DataFrame([{"call_id": t.call_id, "notes": " ".join(t.quality_notes)} for t, _ in processed])

Failures logged: none


,call_id,notes
0,001,2% of turns flagged as low ASR confidence (uncalibrated heuristic).
1,002,
2,003,
3,004,"1% of turns flagged as low ASR confidence (uncalibrated heuristic). Speakers overlap on 45% of the quieter speaker's time: the channels may be out of sync, ..."
4,005,"Speakers overlap on 61% of the quieter speaker's time: the channels may be out of sync, so turn order, the summary, and the sentiment windows are unreliable."
5,006,2% of turns flagged as low ASR confidence (uncalibrated heuristic).
6,007,
7,008,3% of turns flagged as low ASR confidence (uncalibrated heuristic).
8,009,2% of turns flagged as low ASR confidence (uncalibrated heuristic).
9,010,"5% of turns flagged as low ASR confidence (uncalibrated heuristic). Speakers overlap on 31% of the quieter speaker's time: the channels may be out of sync, ..."
